# 02 — Feature Engineering

This notebook builds on the completed EDA and reuses its exported measurements. It adds rolling features, defines eligible observations, and assembles inputs for a future stock-ranking model. Explanatory Markdown is followed by code cells for your implementation.

## Theory before implementation

Two stocks both gained 10% over the last month. One rose steadily; the other swung sharply and now has twice its normal volume. Features describe those differences numerically so a model can investigate their relationship with later outcomes.

A feature is a measured input available at prediction time. Reuse EDA's past 5- and 20-day returns as momentum features. Add rolling volatility to describe recent return variability and relative volume to describe unusual trading activity. These are candidate predictors, not proven signals.

A rolling window advances through each stock's history: it drops the oldest observation and includes a newer one. Unlike a seven-year summary, a rolling feature can be calculated using only the information available on its date. Keep stocks separate and rows chronologically ordered.

Assume predictions happen after today's close. Today's price and volume are then available. Volatility uses 20 returns ending today. Relative volume compares today with the previous 20 volume observations, excluding today. Rolling dollar volume describes recent typical trading value and can support eligibility decisions.

Percentile ranks compare features among eligible peers on the same date. A high percentile means a high feature value; high volatility is not automatically desirable. Rolling calculations introduce missing values until enough history exists. Missing measurements must not be replaced with zero.

The output contains one row per eligible stock-date with four raw inputs, any chosen percentile transformations, and documented definitions. Verification establishes calculation correctness; later validation establishes predictive usefulness.

## Implementation

The following sections reuse EDA columns, create rolling features, verify them, and export the eligible table.

## 1. Load and reuse the EDA dataset

Load the CSV exported from notebook 01 and restore the date column as dates. It already includes daily_return, five_day_return, twenty_day_return, dollar_volume, and their existing return ranks. Reuse these measurements instead of repeating the EDA calculations. Sort the table by symbol and date and confirm the expected columns are present.

The five_day_return and twenty_day_return columns are already 5- and 20-day momentum features. Normalized prices remain useful for visualization but are not part of the initial model input set.

In [10]:

import pandas as pd

from factor_model.config import EDA_DATA_PATH

df = pd.read_csv(EDA_DATA_PATH, parse_dates=["date"])
df.head()

,date,symbol,open,high,low,close,adjusted_close,volume,normalized_price,dollar_volume,daily_return,five_day_return,twenty_day_return,daily_rank,five_day_rank,twenty_day_rank
0,2018-01-02,AAPL,42.540001,43.075001,42.314999,43.064999,40.232380,102223600,100.000000,4.402259e+09,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-03,AAPL,43.132500,43.637501,42.990002,43.057499,40.225365,118071600,99.982563,5.083868e+09,-0.000174,NaN,NaN,9.0,NaN,NaN
2,2018-01-04,AAPL,43.134998,43.367500,43.020000,43.257500,40.412216,89738400,100.446994,3.881859e+09,0.004645,NaN,NaN,5.0,NaN,NaN
3,2018-01-05,AAPL,43.360001,43.842499,43.262501,43.750000,40.872318,94640000,101.590605,4.140500e+09,0.011385,NaN,NaN,3.0,NaN,NaN
4,2018-01-08,AAPL,43.587502,43.902500,43.482498,43.587502,40.720520,82271200,101.213302,3.585996e+09,-0.003714,NaN,NaN,10.0,NaN,NaN


## 2. Define the model inputs and prediction timing

Imagine two stocks both gained 10% over the last month. One moved steadily; the other swung sharply and now has unusually high trading volume. Features give the model numerical descriptions of these differences on each date.

Use four initial inputs: five_day_return, twenty_day_return, volatility_20, and relative_volume_20. The hypothesis is that recent direction, variability, and unusual activity may help predict future relative performance. These inputs are candidates, not proven predictors.

Assume predictions are made after the close of day t. Today's close and volume are available, but tomorrow's observations are not. Keep this timing assumption in your experiment notes. The code cell can hold your explicit list of feature column names.

In [11]:
feature_columns = [
    "five_day_return", 
    "twenty_day_return",
    "volatility_20",
    "relative_volume_20"
]

## 3. Calculate 20-day rolling volatility

In EDA, each stock had one volatility summary for the entire sample. A model needs a value available on each prediction date. For example, Friday's volatility should describe the recent period ending Friday, not the stock's next year of returns.

Create volatility_20 as the rolling sample standard deviation of daily_return over 20 observations within each symbol. Require 20 valid returns and keep the result in daily units. A value of 0.02 means a daily return standard deviation of 2%. This adds a description of recent variability beyond the existing momentum features.

In [12]:
df["volatility_20"] = (
    df.groupby("symbol").daily_return.
    transform(
        lambda x: x.rolling(20, min_periods=20).std()
    )
)

## 4. Calculate relative volume

Ten million shares traded today could be ordinary for AAPL but unusually high for another stock. Compare today's volume with the same stock's own recent baseline.

Create relative_volume_20 = today's volume / mean volume over the previous 20 observations, excluding today. A value of 2 means twice the recent average; 0.5 means half. Shift the baseline within each symbol, require a full window, and handle missing or zero denominators explicitly. Today's numerator is available under the after-close prediction assumption.

In [13]:
average_volume_20 = df.groupby("symbol").volume.transform(lambda x: x.shift(1).rolling(20, min_periods=20).mean())

df["relative_volume_20"] = (
    df.volume / average_volume_20.where(average_volume_20 > 0)
)

## 5. Calculate rolling liquidity

EDA already created dollar_volume = close × volume. Reuse that column to calculate average_dollar_volume_20, a rolling mean over 20 observations within each symbol, including today under the after-close assumption.

This approximates recent typical trading value. Keep it initially as a descriptive field or an input to an eligibility rule, rather than automatically adding it to the four model inputs. A liquidity threshold, if used, must depend on information available by that date. Daily close multiplied by volume is an approximation of traded value.

In [14]:
df["average_dollar_volume_20"] = (
    df.groupby("symbol").dollar_volume
    .transform(
        lambda dollar_volume: dollar_volume
        .rolling(20, min_periods=20).mean()
    )
)

## 6. Handle incomplete history and eligibility

The first observations lack enough history for rolling calculations. Missing volatility or momentum does not mean zero volatility or zero return; it means the measurement is unavailable.

Keep the full table for diagnostics, then create a separate eligible table containing rows with all four required inputs available and finite. Apply any documented liquidity rule here. Count eligible stocks per date before comparing them. Rolling windows count observations, so the EDA daily-coverage check remains necessary before interpreting them as trading-day windows.

In [15]:
import numpy as np

eligible_df = df.loc[
    np.isfinite(df[feature_columns]).all(axis=1)
].copy()

## 7. Compare features within each date

Your EDA ranks already compare past returns across stocks, with rank 1 assigned to the largest return. You can retain those for inspection. For consistent model inputs, create percentile versions of the four selected features within each date of the eligible table.

Use a convention where larger raw values receive larger percentiles and average ranks resolve ties. With ten distinct values, percentiles run from 0.1 to 1.0. A high volatility percentile describes relatively high volatility; it does not imply a better expected return.

This transformation applies the same comparison to momentum, volatility, and relative volume. It uses the stocks available on that date. Any later transformation fitted across time must be fitted on training data only.

In [16]:
eligible_df["momentum_5_percentile"] = (
    eligible_df.groupby("date")["five_day_return"]
    .rank(method="average", ascending=True, pct=True)
)

## 8. Verify values and timing

Reuse the EDA momentum columns, but verify a sample value against its two adjusted-close prices. Check one rolling volatility window and one relative-volume baseline manually or against a small transparent example. Inspect first valid dates, missing counts, extreme values, and distributions.

Recalculate the new features on a shortened copy of the history and compare their earlier values with calculations from the full dataset. Adding future observations must not change earlier features. Check correlations for repeated information. An optional reversal score is simply the negative of a recent return and does not add information when that same return is already included.

In [17]:

aapl = df[df["symbol"] == "AAPL"].sort_values("date")

price_before = aapl.iloc[0]["adjusted_close"]
price_after = aapl.iloc[5]["adjusted_close"]

expected_return = price_after / price_before - 1
stored_return = aapl.iloc[5]["five_day_return"]

print("Calculated from prices:", expected_return)
print("Stored in the table:   ", stored_return)

print("-" * 67)

aapl = df[df["symbol"] == "AAPL"].sort_values("date")

previous_average = aapl.iloc[:20]["volume"].mean()
today_volume = aapl.iloc[20]["volume"]

print("Calculated:", today_volume / previous_average)
print("Stored:    ", aapl.iloc[20]["relative_volume_20"])
# Fixed relative volume

print("-" * 67)

aapl = df[df["symbol"] == "AAPL"].sort_values("date")

expected = aapl.iloc[1:21]["daily_return"].std()
stored = aapl.iloc[20]["volatility_20"]

print("Calculated:", expected)
print("Stored:    ", stored)
# fixed no std() in section 3


print("-" * 67)

aapl = df[df["symbol"] == "AAPL"].sort_values("date")

expected = aapl.iloc[:20]["dollar_volume"].mean()
stored = aapl.iloc[19]["average_dollar_volume_20"]

print("Calculated:", expected)
print("Stored:    ", stored)
# correct on first try

print("-" * 67)

df.groupby("symbol")[feature_columns].agg(
  lambda column: column.isna().sum()
)
# aswell

print("-" * 67)
import numpy as np

assert not np.isinf(df[feature_columns]).any().any(), (
  "Features contain infinite values"
) 
df[feature_columns].dropna().corr()

dates = df["date"].drop_duplicates().sort_values()
cutoff = dates.iloc[len(dates) // 2]

past = (
  df.loc[df["date"] <= cutoff]
    .sort_values(["symbol", "date"])
    .copy()
)
recalculated = (
  past.groupby("symbol")["daily_return"]
    .transform(
      lambda returns: returns.rolling(20, min_periods=20).std()
    )
)

np.testing.assert_allclose(
  past["volatility_20"],
  recalculated,
  equal_nan=True,
)


past_average_volume = (
  past.groupby("symbol")["volume"]
    .transform(
      lambda volume: volume.shift(1)
        .rolling(20, min_periods=20)
        .mean()
    )
)

recalculated = (
  past["volume"]
  / past_average_volume.where(past_average_volume > 0)
)

np.testing.assert_allclose(
  past["relative_volume_20"],
  recalculated,
  equal_nan=True,
)

Calculated from prices: 0.012016960413218314
Stored in the table:    0.0120169604132183
-------------------------------------------------------------------
Calculated: 1.035678383547207
Stored:     1.035678383547207
-------------------------------------------------------------------
Calculated: 0.009461566527878496
Stored:     0.009461566527878497
-------------------------------------------------------------------
Calculated: 5452773257.114792
Stored:     5452773257.114792
-------------------------------------------------------------------
-------------------------------------------------------------------


## 9. Assemble the feature dataset

Produce one row per eligible (date, symbol), keeping the four raw inputs and their percentile versions clearly named. Preserve keys and useful diagnostic columns, and choose explicitly whether the initial model will use raw values or percentile values. It need not use both versions of every feature.

The target belongs to a later stage: for example, a stock's relative return rank over the next five trading days. Today's features describe what was known; the target describes what subsequently happened. Keep their calculations separate and evaluate predictions with chronological splits.

Save the feature dataset to a separate processed-data file so the EDA export stays available. Add its shared path to config.py when implementing the export.

In [18]:
from factor_model.config import FTE_DATA_PATH

eligible_df.to_csv(FTE_DATA_PATH, index=False)

## Feature engineering findings

Predictions are assumed to be made after the market close. Each feature uses information available through its date; future prices are excluded from model inputs.

The four initial raw inputs are five_day_return and twenty_day_return, reused from EDA; volatility_20, the sample standard deviation of the latest 20 daily returns including today; and relative_volume_20, today's volume divided by the preceding 20 observations' average volume, excluding today. average_dollar_volume_20 describes recent trading activity and is retained as a diagnostic field. No liquidity threshold is currently applied.

Manual sample checks for AAPL matched the stored 5-day return, relative volume, volatility, and rolling dollar volume. Expected missing-history counts were confirmed during the walkthrough. The saved verification cell rejects infinite feature values and checks that earlier volatility and relative-volume values agree with recalculations using shortened history. These checks establish specific calculation properties; they do not prove predictive usefulness or validate the entire pipeline.

Eligibility requires all four raw inputs to be available and finite. The export contains 17,410 rows across 1,741 observed dates, from January 31, 2018 through December 31, 2024. Every retained date contains 10 unique symbols, and the exported (date, symbol) keys are unique. The initial 20 observations per stock are excluded because the full feature history is unavailable.

The dataset is saved to FTE_DATA_PATH, currently data/processed/fte_data.csv. It contains the raw inputs but no percentile columns: momentum_5_percentile was calculated on nndf, while the export uses a separate eligible_df. The initial target notebook can use the raw-feature export; percentile preprocessing remains unfinished if that representation is chosen later.

The next stage creates separate forward-return targets and joins them by (date, symbol). Whether the features help predict future stock rankings must be evaluated using chronological validation. The small, manually selected surviving-company universe remains a limitation.